In [0]:
from pyspark.sql import Row

CATALOG = "power_prices"

# ---------------------------------------------------------------
# 1. dim_bidding_zone: ENTSO-E bidding zones (EIC codes)
# ---------------------------------------------------------------
zones = [
    # zone_code, eic_code, zone_name, country_code, country_name, region, timezone
    ("AT",      "10YAT-APG------L", "Austria",                   "AT", "Austria",        "Central West",  "Europe/Vienna"),
    ("BE",      "10YBE----------2", "Belgium",                   "BE", "Belgium",        "Central West",  "Europe/Brussels"),
    ("CH",      "10YCH-SWISSGRIDZ", "Switzerland",               "CH", "Switzerland",    "Central West",  "Europe/Zurich"),
    ("DE_LU",   "10Y1001A1001A82H", "Germany-Luxembourg",        "DE", "Germany",        "Central West",  "Europe/Berlin"),
    ("FR",      "10YFR-RTE------C", "France",                    "FR", "France",         "Central West",  "Europe/Paris"),
    ("NL",      "10YNL----------L", "Netherlands",               "NL", "Netherlands",    "Central West",  "Europe/Amsterdam"),
    ("DK_1",    "10YDK-1--------W", "Denmark West (DK1)",        "DK", "Denmark",        "Nordics",       "Europe/Copenhagen"),
    ("DK_2",    "10YDK-2--------M", "Denmark East (DK2)",        "DK", "Denmark",        "Nordics",       "Europe/Copenhagen"),
    ("FI",      "10YFI-1--------U", "Finland",                   "FI", "Finland",        "Nordics",       "Europe/Helsinki"),
    ("NO_1",    "10YNO-1--------2", "Norway Southeast (NO1)",    "NO", "Norway",         "Nordics",       "Europe/Oslo"),
    ("NO_2",    "10YNO-2--------T", "Norway Southwest (NO2)",    "NO", "Norway",         "Nordics",       "Europe/Oslo"),
    ("NO_3",    "10YNO-3--------J", "Norway Central (NO3)",      "NO", "Norway",         "Nordics",       "Europe/Oslo"),
    ("NO_4",    "10YNO-4--------9", "Norway North (NO4)",        "NO", "Norway",         "Nordics",       "Europe/Oslo"),
    ("NO_5",    "10Y1001A1001A48H", "Norway West (NO5)",         "NO", "Norway",         "Nordics",       "Europe/Oslo"),
    ("SE_1",    "10Y1001A1001A44P", "Sweden North (SE1)",        "SE", "Sweden",         "Nordics",       "Europe/Stockholm"),
    ("SE_2",    "10Y1001A1001A45N", "Sweden North-Central (SE2)","SE", "Sweden",         "Nordics",       "Europe/Stockholm"),
    ("SE_3",    "10Y1001A1001A46L", "Sweden South-Central (SE3)","SE", "Sweden",         "Nordics",       "Europe/Stockholm"),
    ("SE_4",    "10Y1001A1001A47J", "Sweden South (SE4)",        "SE", "Sweden",         "Nordics",       "Europe/Stockholm"),
    ("EE",      "10Y1001A1001A39I", "Estonia",                   "EE", "Estonia",        "Baltics",       "Europe/Tallinn"),
    ("LT",      "10YLT-1001A0008Q", "Lithuania",                 "LT", "Lithuania",      "Baltics",       "Europe/Vilnius"),
    ("LV",      "10YLV-1001A00074", "Latvia",                    "LV", "Latvia",         "Baltics",       "Europe/Riga"),
    ("ES",      "10YES-REE------0", "Spain",                     "ES", "Spain",          "Iberia",        "Europe/Madrid"),
    ("PT",      "10YPT-REN------W", "Portugal",                  "PT", "Portugal",       "Iberia",        "Europe/Lisbon"),
    ("IT_NORD", "10Y1001A1001A73I", "Italy North",               "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("IT_CNOR", "10Y1001A1001A70O", "Italy Centre-North",        "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("IT_CSUD", "10Y1001A1001A71M", "Italy Centre-South",        "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("IT_SUD",  "10Y1001A1001A788", "Italy South",               "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("IT_CALA", "10Y1001C--00096J", "Italy Calabria",            "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("IT_SICI", "10Y1001A1001A75E", "Italy Sicily",              "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("IT_SARD", "10Y1001A1001A74G", "Italy Sardinia",            "IT", "Italy",          "Italy",         "Europe/Rome"),
    ("CZ",      "10YCZ-CEPS-----N", "Czechia",                   "CZ", "Czechia",        "Central East",  "Europe/Prague"),
    ("HU",      "10YHU-MAVIR----U", "Hungary",                   "HU", "Hungary",        "Central East",  "Europe/Budapest"),
    ("PL",      "10YPL-AREA-----S", "Poland",                    "PL", "Poland",         "Central East",  "Europe/Warsaw"),
    ("SK",      "10YSK-SEPS-----K", "Slovakia",                  "SK", "Slovakia",       "Central East",  "Europe/Bratislava"),
    ("SI",      "10YSI-ELES-----O", "Slovenia",                  "SI", "Slovenia",       "Central East",  "Europe/Ljubljana"),
    ("HR",      "10YHR-HEP------M", "Croatia",                   "HR", "Croatia",        "Central East",  "Europe/Zagreb"),
    ("RO",      "10YRO-TEL------P", "Romania",                   "RO", "Romania",        "South East",    "Europe/Bucharest"),
    ("BG",      "10YCA-BULGARIA-R", "Bulgaria",                  "BG", "Bulgaria",       "South East",    "Europe/Sofia"),
    ("GR",      "10YGR-HTSO-----Y", "Greece",                    "GR", "Greece",         "South East",    "Europe/Athens"),
    ("RS",      "10YCS-SERBIATSOV", "Serbia",                    "RS", "Serbia",         "South East",    "Europe/Belgrade"),
    ("IE_SEM",  "10Y1001A1001A59C", "Ireland (SEM)",             "IE", "Ireland",        "Ireland",       "Europe/Dublin"),
]

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.gold.dim_bidding_zone (
  zone_code     STRING  NOT NULL COMMENT 'Short zone code used across the model, e.g. DE_LU',
  eic_code      STRING  NOT NULL COMMENT 'ENTSO-E Energy Identification Code used in API calls',
  zone_name     STRING  NOT NULL COMMENT 'Plain-words zone name for the dashboard',
  country_code  STRING  NOT NULL COMMENT 'ISO 3166 alpha-2 country code',
  country_name  STRING  NOT NULL COMMENT 'Country name',
  region        STRING  NOT NULL COMMENT 'Market region grouping for the dashboard',
  timezone      STRING  NOT NULL COMMENT 'IANA time zone for local-hour analysis (DST-aware)',
  is_active     BOOLEAN NOT NULL COMMENT 'True if the zone is loaded by the pipeline'
)
COMMENT 'Bidding zones covered by the project, with ENTSO-E EIC codes and local time zones'
""")

zone_df = spark.createDataFrame(
    [Row(zone_code=z[0], eic_code=z[1], zone_name=z[2], country_code=z[3],
         country_name=z[4], region=z[5], timezone=z[6], is_active=True) for z in zones]
)
zone_df.write.insertInto(f"{CATALOG}.gold.dim_bidding_zone", overwrite=True)

# ---------------------------------------------------------------
# 2. dim_fuel_type: ENTSO-E production types (psrType codes)
# ---------------------------------------------------------------
fuels = [
    # psr_code, fuel_name, fuel_group, energy_class, is_variable_renewable, sort_order
    ("B14", "Nuclear",                          "Nuclear",  "Nuclear",   False,  1),
    ("B02", "Fossil Brown coal/Lignite",        "Lignite",  "Fossil",    False,  2),
    ("B05", "Fossil Hard coal",                 "Coal",     "Fossil",    False,  3),
    ("B04", "Fossil Gas",                       "Gas",      "Fossil",    False,  4),
    ("B03", "Fossil Coal-derived gas",          "Gas",      "Fossil",    False,  5),
    ("B06", "Fossil Oil",                       "Oil",      "Fossil",    False,  6),
    ("B07", "Fossil Oil shale",                 "Oil",      "Fossil",    False,  7),
    ("B08", "Fossil Peat",                      "Other",    "Fossil",    False,  8),
    ("B11", "Hydro Run-of-river and poundage",  "Hydro",    "Renewable", False,  9),
    ("B12", "Hydro Water Reservoir",            "Hydro",    "Renewable", False, 10),
    ("B19", "Wind Onshore",                     "Wind",     "Renewable", True,  11),
    ("B18", "Wind Offshore",                    "Wind",     "Renewable", True,  12),
    ("B16", "Solar",                            "Solar",    "Renewable", True,  13),
    ("B01", "Biomass",                          "Biomass",  "Renewable", False, 14),
    ("B09", "Geothermal",                       "Other",    "Renewable", False, 15),
    ("B13", "Marine",                           "Other",    "Renewable", False, 16),
    ("B15", "Other renewable",                  "Other",    "Renewable", False, 17),
    ("B17", "Waste",                            "Other",    "Other",     False, 18),
    ("B20", "Other",                            "Other",    "Other",     False, 19),
    ("B10", "Hydro Pumped Storage",             "Storage",  "Storage",   False, 20),
    ("B25", "Energy storage",                   "Storage",  "Storage",   False, 21),
]

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.gold.dim_fuel_type (
  psr_code               STRING  NOT NULL COMMENT 'ENTSO-E production type code (psrType), e.g. B16',
  fuel_name              STRING  NOT NULL COMMENT 'ENTSO-E production type name',
  fuel_group             STRING  NOT NULL COMMENT 'Dashboard grouping: Nuclear, Lignite, Coal, Gas, Oil, Hydro, Wind, Solar, Biomass, Storage, Other',
  energy_class           STRING  NOT NULL COMMENT 'Renewable, Fossil, Nuclear, Storage or Other',
  is_variable_renewable  BOOLEAN NOT NULL COMMENT 'True for weather-driven output (wind and solar)',
  sort_order             INT     NOT NULL COMMENT 'Display order, roughly baseload to variable'
)
COMMENT 'Generation fuel types from ENTSO-E psrType codes, grouped for the dashboard'
""")

fuel_df = spark.createDataFrame(
    [Row(psr_code=f[0], fuel_name=f[1], fuel_group=f[2], energy_class=f[3],
         is_variable_renewable=f[4], sort_order=f[5]) for f in fuels]
)
fuel_df.write.insertInto(f"{CATALOG}.gold.dim_fuel_type", overwrite=True)

# ---------------------------------------------------------------
# 3. Quality checks
# ---------------------------------------------------------------
z = spark.table(f"{CATALOG}.gold.dim_bidding_zone")
f = spark.table(f"{CATALOG}.gold.dim_fuel_type")
assert z.count() == z.select("zone_code").distinct().count(), "Duplicate zone_code"
assert z.count() == z.select("eic_code").distinct().count(), "Duplicate eic_code"
assert f.count() == f.select("psr_code").distinct().count(), "Duplicate psr_code"
print(f"dim_bidding_zone: {z.count()} zones | dim_fuel_type: {f.count()} fuel types | checks passed")

display(z.groupBy("region").count().orderBy("region"))
display(f.groupBy("fuel_group", "energy_class").count().orderBy("fuel_group"))